# 1.Preparacion_Ambiente — Olist — PRODUCCIÓN

**Ruta:** `proceso/1.Preparacion_Ambiente` · **Lenguaje:** PySpark + sentencias DDL en `spark.sql`.

Crea catálogo, 4 esquemas y las **30 tablas Delta externas** de Olist en `/prod/` sin alterar DEV. Parte de las definiciones de las tablas Olist ya verificadas en el proyecto. **No crea ni modifica las tablas Raw ni sus CSV.**

Requisitos: la credencial de almacenamiento `credential`, las External Locations y los permisos sobre ADLS Gen2 ya deben estar configurados y accesibles desde PROD. El job se ejecuta con una identidad autorizada para crear catálogo, esquemas y tablas.


In [ ]:
dbutils.widgets.text("catalogo", "catalog_prod")
catalogo = dbutils.widgets.get("catalogo").strip()
if catalogo != "catalog_prod":
    raise ValueError("Este notebook solo prepara catalog_prod")
storage = "adlsmartdata08100"
print("Preparación Olist PROD: ",catalogo)


## Crear catálogo y esquemas (idempotente)

In [ ]:
# Se aprovecha la External Location 'exlt-metastore' del proyecto.
# La creación del catálogo puede requerir privilegios CREATE CATALOG y
# CREATE MANAGED STORAGE sobre la ubicación externa.
spark.sql("""
CREATE CATALOG IF NOT EXISTS catalog_prod
MANAGED LOCATION 'abfss://metastore@adlsmartdata08100.dfs.core.windows.net/prod'
""")
for esquema in ["raw", "bronze", "silver", "golden"]:
    spark.sql(f"CREATE SCHEMA IF NOT EXISTS {catalogo}.{esquema}")
    print(f"Esquema preparado: {catalogo}.{esquema}")


## Crear tablas Bronze (9)

In [ ]:
# DDL tomadas de las definiciones Olist ya verificadas en DEV.
# CREATE TABLE IF NOT EXISTS permite reejecutar la preparación.
consultas = ["CREATE TABLE IF NOT EXISTS catalog_prod.bronze.orders (\n    order_id STRING,\n    customer_id STRING,\n    order_status STRING,\n    order_purchase_timestamp STRING,\n    order_approved_at STRING,\n    order_delivered_carrier_date STRING,\n    order_delivered_customer_date STRING,\n    order_estimated_delivery_date STRING\n)\nUSING DELTA\nLOCATION 'abfss://bronze@adlsmartdata08100.dfs.core.windows.net/prod/orders';", "CREATE TABLE IF NOT EXISTS catalog_prod.bronze.order_items (\n    order_id STRING,\n    order_item_id STRING,\n    product_id STRING,\n    seller_id STRING,\n    shipping_limit_date STRING,\n    price STRING,\n    freight_value STRING\n)\nUSING DELTA\nLOCATION 'abfss://bronze@adlsmartdata08100.dfs.core.windows.net/prod/order_items';", "CREATE TABLE IF NOT EXISTS catalog_prod.bronze.customers (\n    customer_id STRING,\n    customer_unique_id STRING,\n    customer_zip_code_prefix STRING,\n    customer_city STRING,\n    customer_state STRING\n)\nUSING DELTA\nLOCATION 'abfss://bronze@adlsmartdata08100.dfs.core.windows.net/prod/customers';", "CREATE TABLE IF NOT EXISTS catalog_prod.bronze.products (\n    product_id STRING,\n    product_category_name STRING,\n    product_name_lenght STRING,\n    product_description_lenght STRING,\n    product_photos_qty STRING,\n    product_weight_g STRING,\n    product_length_cm STRING,\n    product_height_cm STRING,\n    product_width_cm STRING\n)\nUSING DELTA\nLOCATION 'abfss://bronze@adlsmartdata08100.dfs.core.windows.net/prod/products';", "CREATE TABLE IF NOT EXISTS catalog_prod.bronze.sellers (\n    seller_id STRING,\n    seller_zip_code_prefix STRING,\n    seller_city STRING,\n    seller_state STRING\n)\nUSING DELTA\nLOCATION 'abfss://bronze@adlsmartdata08100.dfs.core.windows.net/prod/sellers';", "CREATE TABLE IF NOT EXISTS catalog_prod.bronze.order_payments (\n    order_id STRING,\n    payment_sequential STRING,\n    payment_type STRING,\n    payment_installments STRING,\n    payment_value STRING\n)\nUSING DELTA\nLOCATION 'abfss://bronze@adlsmartdata08100.dfs.core.windows.net/prod/order_payments';", "CREATE TABLE IF NOT EXISTS catalog_prod.bronze.order_reviews (\n    review_id STRING,\n    order_id STRING,\n    review_score STRING,\n    review_comment_title STRING,\n    review_comment_message STRING,\n    review_creation_date STRING,\n    review_answer_timestamp STRING\n)\nUSING DELTA\nLOCATION 'abfss://bronze@adlsmartdata08100.dfs.core.windows.net/prod/order_reviews';", "CREATE TABLE IF NOT EXISTS catalog_prod.bronze.geolocation (\n    geolocation_zip_code_prefix STRING,\n    geolocation_lat STRING,\n    geolocation_lng STRING,\n    geolocation_city STRING,\n    geolocation_state STRING\n)\nUSING DELTA\nLOCATION 'abfss://bronze@adlsmartdata08100.dfs.core.windows.net/prod/geolocation';", "CREATE TABLE IF NOT EXISTS catalog_prod.bronze.category_translation (\n    product_category_name STRING,\n    product_category_name_english STRING\n)\nUSING DELTA\nLOCATION 'abfss://bronze@adlsmartdata08100.dfs.core.windows.net/prod/category_translation';"]
for sentencia in consultas:
    spark.sql(sentencia)
    print("OK:", sentencia.split("(")[0].strip())


## Crear tablas Silver (9)

In [ ]:
# DDL tomadas de las definiciones Olist ya verificadas en DEV.
# CREATE TABLE IF NOT EXISTS permite reejecutar la preparación.
consultas = ["CREATE TABLE IF NOT EXISTS catalog_prod.silver.pedidos (\n    id_pedido STRING,\n    id_cliente STRING,\n    estado_pedido STRING,\n    fecha_compra TIMESTAMP,\n    fecha_aprobacion TIMESTAMP,\n    fecha_entrega_transportista TIMESTAMP,\n    fecha_entrega_cliente TIMESTAMP,\n    fecha_entrega_estimada TIMESTAMP\n)\nUSING DELTA\nLOCATION 'abfss://silver@adlsmartdata08100.dfs.core.windows.net/prod/pedidos';", "CREATE TABLE IF NOT EXISTS catalog_prod.silver.detalle_ventas (\n    id_pedido STRING,\n    id_item INT,\n    id_producto STRING,\n    id_vendedor STRING,\n    fecha_limite_envio TIMESTAMP,\n    precio DECIMAL(18,2),\n    valor_flete DECIMAL(18,2)\n)\nUSING DELTA\nLOCATION 'abfss://silver@adlsmartdata08100.dfs.core.windows.net/prod/detalle_ventas';", "CREATE TABLE IF NOT EXISTS catalog_prod.silver.clientes (\n    id_cliente STRING,\n    id_cliente_unico STRING,\n    codigo_postal STRING,\n    ciudad STRING,\n    estado STRING\n)\nUSING DELTA\nLOCATION 'abfss://silver@adlsmartdata08100.dfs.core.windows.net/prod/clientes';", "CREATE TABLE IF NOT EXISTS catalog_prod.silver.productos (\n    id_producto STRING,\n    categoria STRING,\n    longitud_nombre INT,\n    longitud_descripcion INT,\n    cantidad_fotos INT,\n    peso_g DOUBLE,\n    largo_cm DOUBLE,\n    alto_cm DOUBLE,\n    ancho_cm DOUBLE\n)\nUSING DELTA\nLOCATION 'abfss://silver@adlsmartdata08100.dfs.core.windows.net/prod/productos';", "CREATE TABLE IF NOT EXISTS catalog_prod.silver.vendedores (\n    id_vendedor STRING,\n    codigo_postal STRING,\n    ciudad STRING,\n    estado STRING\n)\nUSING DELTA\nLOCATION 'abfss://silver@adlsmartdata08100.dfs.core.windows.net/prod/vendedores';", "CREATE TABLE IF NOT EXISTS catalog_prod.silver.pagos (\n    id_pedido STRING,\n    secuencia_pago INT,\n    metodo_pago STRING,\n    cuotas INT,\n    importe_pago DECIMAL(18,2)\n)\nUSING DELTA\nLOCATION 'abfss://silver@adlsmartdata08100.dfs.core.windows.net/prod/pagos';", "CREATE TABLE IF NOT EXISTS catalog_prod.silver.resenas (\n    id_resena STRING,\n    id_pedido STRING,\n    calificacion INT,\n    titulo STRING,\n    comentario STRING,\n    fecha_creacion TIMESTAMP,\n    fecha_respuesta TIMESTAMP\n)\nUSING DELTA\nLOCATION 'abfss://silver@adlsmartdata08100.dfs.core.windows.net/prod/resenas';", "CREATE TABLE IF NOT EXISTS catalog_prod.silver.ubicaciones (\n    codigo_postal STRING,\n    ciudad STRING,\n    estado STRING,\n    latitud DOUBLE,\n    longitud DOUBLE,\n    registros_geograficos BIGINT\n)\nUSING DELTA\nLOCATION 'abfss://silver@adlsmartdata08100.dfs.core.windows.net/prod/ubicaciones';", "CREATE TABLE IF NOT EXISTS catalog_prod.silver.categorias (\n    categoria STRING,\n    categoria_ingles STRING\n)\nUSING DELTA\nLOCATION 'abfss://silver@adlsmartdata08100.dfs.core.windows.net/prod/categorias';"]
for sentencia in consultas:
    spark.sql(sentencia)
    print("OK:", sentencia.split("(")[0].strip())


## Crear tablas Golden (12)

In [ ]:
# DDL tomadas de las definiciones Olist ya verificadas en DEV.
# CREATE TABLE IF NOT EXISTS permite reejecutar la preparación.
consultas = ["CREATE TABLE IF NOT EXISTS catalog_prod.golden.dim_fecha (\n    id_fecha INT,\n    fecha DATE,\n    anio INT,\n    trimestre INT,\n    mes INT,\n    nombre_mes STRING,\n    semana_anio INT,\n    dia_mes INT,\n    dia_semana INT,\n    nombre_dia STRING\n)\nUSING DELTA\nLOCATION 'abfss://golden@adlsmartdata08100.dfs.core.windows.net/prod/dim_fecha';", "CREATE TABLE IF NOT EXISTS catalog_prod.golden.dim_cliente (\n    id_cliente_unico STRING\n)\nUSING DELTA\nLOCATION 'abfss://golden@adlsmartdata08100.dfs.core.windows.net/prod/dim_cliente';", "CREATE TABLE IF NOT EXISTS catalog_prod.golden.dim_producto (\n    id_producto STRING,\n    longitud_nombre INT,\n    longitud_descripcion INT,\n    cantidad_fotos INT,\n    peso_g DOUBLE,\n    largo_cm DOUBLE,\n    alto_cm DOUBLE,\n    ancho_cm DOUBLE\n)\nUSING DELTA\nLOCATION 'abfss://golden@adlsmartdata08100.dfs.core.windows.net/prod/dim_producto';", "CREATE TABLE IF NOT EXISTS catalog_prod.golden.dim_categoria (\n    id_categoria STRING,\n    categoria_ingles STRING\n)\nUSING DELTA\nLOCATION 'abfss://golden@adlsmartdata08100.dfs.core.windows.net/prod/dim_categoria';", "CREATE TABLE IF NOT EXISTS catalog_prod.golden.dim_vendedor (\n    id_vendedor STRING,\n    codigo_postal STRING,\n    ciudad STRING,\n    estado STRING\n)\nUSING DELTA\nLOCATION 'abfss://golden@adlsmartdata08100.dfs.core.windows.net/prod/dim_vendedor';", "CREATE TABLE IF NOT EXISTS catalog_prod.golden.dim_geografia (\n    codigo_postal STRING,\n    ciudad STRING,\n    estado STRING,\n    latitud DOUBLE,\n    longitud DOUBLE,\n    registros_geograficos BIGINT\n)\nUSING DELTA\nLOCATION 'abfss://golden@adlsmartdata08100.dfs.core.windows.net/prod/dim_geografia';", "CREATE TABLE IF NOT EXISTS catalog_prod.golden.dim_metodo_pago (\n    metodo_pago STRING\n)\nUSING DELTA\nLOCATION 'abfss://golden@adlsmartdata08100.dfs.core.windows.net/prod/dim_metodo_pago';", "CREATE TABLE IF NOT EXISTS catalog_prod.golden.dim_estado_pedido (\n    estado_pedido STRING\n)\nUSING DELTA\nLOCATION 'abfss://golden@adlsmartdata08100.dfs.core.windows.net/prod/dim_estado_pedido';", "CREATE TABLE IF NOT EXISTS catalog_prod.golden.fact_ventas (\n    id_pedido STRING,\n    id_item INT,\n    id_fecha INT,\n    id_cliente_unico STRING,\n    id_producto STRING,\n    id_categoria STRING,\n    id_vendedor STRING,\n    codigo_postal_cliente STRING,\n    estado_pedido STRING,\n    precio DECIMAL(18,2),\n    valor_flete DECIMAL(18,2),\n    cantidad_articulos INT\n)\nUSING DELTA\nLOCATION 'abfss://golden@adlsmartdata08100.dfs.core.windows.net/prod/fact_ventas';", "CREATE TABLE IF NOT EXISTS catalog_prod.golden.fact_pedidos (\n    id_pedido STRING,\n    id_fecha INT,\n    id_cliente_unico STRING,\n    codigo_postal_cliente STRING,\n    estado_pedido STRING,\n    fecha_compra TIMESTAMP,\n    fecha_entrega_cliente TIMESTAMP,\n    fecha_entrega_estimada TIMESTAMP,\n    dias_entrega DOUBLE,\n    dias_retraso DOUBLE,\n    entrega_a_tiempo INT,\n    cantidad_articulos BIGINT,\n    importe_articulos DECIMAL(18,2),\n    importe_flete DECIMAL(18,2),\n    importe_total DECIMAL(18,2),\n    tiene_detalle INT,\n    indicador_pedido INT\n)\nUSING DELTA\nLOCATION 'abfss://golden@adlsmartdata08100.dfs.core.windows.net/prod/fact_pedidos';", "CREATE TABLE IF NOT EXISTS catalog_prod.golden.fact_pagos (\n    id_pedido STRING,\n    secuencia_pago INT,\n    id_fecha INT,\n    id_cliente_unico STRING,\n    metodo_pago STRING,\n    estado_pedido STRING,\n    importe_pago DECIMAL(18,2),\n    cuotas INT\n)\nUSING DELTA\nLOCATION 'abfss://golden@adlsmartdata08100.dfs.core.windows.net/prod/fact_pagos';", "CREATE TABLE IF NOT EXISTS catalog_prod.golden.fact_resenas (\n    id_resena STRING,\n    id_pedido STRING,\n    id_fecha INT,\n    id_cliente_unico STRING,\n    estado_pedido STRING,\n    calificacion INT,\n    fecha_creacion TIMESTAMP,\n    fecha_respuesta TIMESTAMP,\n    entrega_a_tiempo INT\n)\nUSING DELTA\nLOCATION 'abfss://golden@adlsmartdata08100.dfs.core.windows.net/prod/fact_resenas';"]
for sentencia in consultas:
    spark.sql(sentencia)
    print("OK:", sentencia.split("(")[0].strip())


## Comprobación final

In [ ]:
tablas = {"bronze": 9, "silver": 9, "golden": 12}
for capa, esperado in tablas.items():
    visibles = spark.sql(f"SHOW TABLES IN {catalogo}.{capa}").count()
    if visibles < esperado:
        raise ValueError(f"{capa}: se esperaban al menos {esperado} tablas, encontradas {visibles}")
    print(f"OK - {catalogo}.{capa}: {visibles} tablas")
print("Preparación del ambiente PROD completada")
